# Setup + Load

In [111]:
import duckdb
import pandas as pd

PARQUET_FILE = "../fingerprints.parquet"
TARGET = "takedown"

In [112]:
con = duckdb.connect()

res = con.execute(
    "SELECT COUNT(*) AS n_rows FROM read_parquet(?)",
    [PARQUET_FILE],
).df()

res

,n_rows
0,193556


In [113]:
df = con.execute(
    "SELECT * FROM read_parquet(?) LIMIT 193556",
    [PARQUET_FILE],
).df()

In [114]:
list(df.columns)

["('create', 'app.bsky.feed.like')",
 "('create', 'app.bsky.feed.post')",
 "('create', 'app.bsky.feed.repost')",
 "('create', 'app.bsky.graph.block')",
 "('create', 'app.bsky.graph.follow')",
 "('delete', 'app.bsky.feed.like')",
 "('delete', 'app.bsky.feed.post')",
 "('delete', 'app.bsky.feed.repost')",
 "('delete', 'app.bsky.graph.block')",
 "('delete', 'app.bsky.graph.follow')",
 "('derived', 'blocked')",
 "('derived', 'blocker')",
 "('derived', 'followed')",
 "('derived', 'follower')",
 "('derived', 'liked')",
 "('derived', 'liker')",
 "('derived', 'replied_parent')",
 "('derived', 'replier_parent')",
 "('derived', 'reposted')",
 "('derived', 'reposter')",
 "('did', '')",
 "('domain', 'bias_center')",
 "('domain', 'bias_conspiracy')",
 "('domain', 'bias_entropy')",
 "('domain', 'bias_extreme-left')",
 "('domain', 'bias_extreme-right')",
 "('domain', 'bias_left')",
 "('domain', 'bias_left-center')",
 "('domain', 'bias_pro-science')",
 "('domain', 'bias_right')",
 "('domain', 'bias_ri

In [115]:
import re
import ast

def clean_col(c):
    # caso indice inutile
    if c == "__index_level_0__":
        return "row_index"

    new = c

    # parse delle tuple stringa
    if isinstance(c, str) and c.startswith("(") and c.endswith(")"):
        try:
            t = ast.literal_eval(c)
            if isinstance(t, tuple) and len(t) == 2:
                left = str(t[0]).strip()
                right = str(t[1]).strip()

                if right == "":
                    new = left
                else:
                    new = f"{left}__{right}"
        except:
            pass

    # normalizzazione
    new = new.lower()
    new = re.sub(r"[.\-\s/]+", "_", new)   # sostituisce ., -, spazio, /
    new = re.sub(r"[^a-z0-9_]", "", new)  # rimuove caratteri strani
    new = re.sub(r"_+", "_", new)         # comprime underscore multipli
    new = new.strip("_")

    return new

# applica rename
rename_map = {col: clean_col(col) for col in df.columns}
df = df.rename(columns=rename_map)

# stampa mapping per controllo
for old, new in rename_map.items():
    print(f"{old} -> {new}")

# verifica finale
print("\nColumns cleaned:")
print(df.columns.tolist())

('create', 'app.bsky.feed.like') -> create_app_bsky_feed_like
('create', 'app.bsky.feed.post') -> create_app_bsky_feed_post
('create', 'app.bsky.feed.repost') -> create_app_bsky_feed_repost
('create', 'app.bsky.graph.block') -> create_app_bsky_graph_block
('create', 'app.bsky.graph.follow') -> create_app_bsky_graph_follow
('delete', 'app.bsky.feed.like') -> delete_app_bsky_feed_like
('delete', 'app.bsky.feed.post') -> delete_app_bsky_feed_post
('delete', 'app.bsky.feed.repost') -> delete_app_bsky_feed_repost
('delete', 'app.bsky.graph.block') -> delete_app_bsky_graph_block
('delete', 'app.bsky.graph.follow') -> delete_app_bsky_graph_follow
('derived', 'blocked') -> derived_blocked
('derived', 'blocker') -> derived_blocker
('derived', 'followed') -> derived_followed
('derived', 'follower') -> derived_follower
('derived', 'liked') -> derived_liked
('derived', 'liker') -> derived_liker
('derived', 'replied_parent') -> derived_replied_parent
('derived', 'replier_parent') -> derived_replier

# Analisy of Deletion of blocks

In [116]:
# Questa sezione analizza la relazione tra blocchi creati e blocchi rimossi a livello account.
# 
# (1) Statistiche aggregate:
# - misura il volume medio di blocchi creati, rimossi e il saldo netto
# - calcola il tasso medio di rimozione (quota di blocchi eliminati rispetto a quelli creati)
# - stima quanto sono diffusi i casi di creazione, rimozione e presenza di entrambi
# → serve per capire se la rimozione dei blocchi è un fenomeno marginale o rilevante nel dataset
#
# (2) Distribuzione del delete rate:
# - suddivide gli account in gruppi in base alla quota di blocchi rimossi
# - permette di osservare se la non-definitività dei blocchi è concentrata in pochi casi o diffusa
# → serve per valutare se il numero di blocchi creati può essere trattato come feature stabile
#    oppure se è necessario considerare anche rimozioni (es. saldo netto o tasso di delete)

In [117]:
# (1)
res = con.execute("""
SELECT
    COUNT(*) AS n_accounts,

    AVG(create_app_bsky_graph_block) AS avg_created,
    AVG(delete_app_bsky_graph_block) AS avg_deleted,
    AVG(create_app_bsky_graph_block - delete_app_bsky_graph_block) AS avg_net,

    AVG(
        CASE
            WHEN create_app_bsky_graph_block > 0
            THEN delete_app_bsky_graph_block * 1.0 / create_app_bsky_graph_block
            ELSE NULL
        END
    ) AS avg_delete_rate,

    SUM(create_app_bsky_graph_block > 0) * 1.0 / COUNT(*) AS pct_with_created,
    SUM(delete_app_bsky_graph_block > 0) * 1.0 / COUNT(*) AS pct_with_deleted,
    SUM(
        create_app_bsky_graph_block > 0
        AND delete_app_bsky_graph_block > 0
    ) * 1.0 / COUNT(*) AS pct_with_both,

    SUM(delete_app_bsky_graph_block > create_app_bsky_graph_block)
        AS n_anomalies

FROM df
""").df()

res

,n_accounts,avg_created,avg_deleted,avg_net,avg_delete_rate,pct_with_created,pct_with_deleted,pct_with_both,n_anomalies
0,193556,9.798012,0.485239,9.312773,0.078711,0.362376,0.074041,0.072987,473.0


In [118]:
# (2)
# La query 3 suddivide gli account in gruppi in base alla quota di blocchi rimossi rispetto a quelli creati (delete rate), 
# distinguendo tra chi non crea blocchi(no_created), chi li crea ma non li rimuove(no_deleted), e chi li rimuove in misura bassa, media o alta. 
# Questo permette di osservare se la rimozione dei blocchi è un fenomeno raro e marginale oppure diffuso tra gli utenti. 
res = con.execute("""
SELECT
    CASE
        WHEN create_app_bsky_graph_block = 0 THEN 'no_created'
        WHEN delete_app_bsky_graph_block = 0 THEN 'no_deleted'
        WHEN delete_app_bsky_graph_block * 1.0 / create_app_bsky_graph_block <= 0.25 THEN 'low'
        WHEN delete_app_bsky_graph_block * 1.0 / create_app_bsky_graph_block <= 0.5 THEN 'medium'
        WHEN delete_app_bsky_graph_block * 1.0 / create_app_bsky_graph_block <= 1 THEN 'high'
        ELSE 'gt_100%'
    END AS bucket,
    COUNT(*) AS n
FROM df
GROUP BY 1
ORDER BY n DESC
""").df()

res

,bucket,n
0,no_created,123416
1,no_deleted,56013
2,low,8359
3,high,2894
4,medium,2605
5,gt_100%,269


# How much the esposition is related to blocking dynamics?

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW exposure_base AS
SELECT
    *,

    -- exposure prodotta: attività totale
    (
        COALESCE(create_app_bsky_feed_post, 0) +
        COALESCE(create_app_bsky_feed_like, 0) +
        COALESCE(create_app_bsky_feed_repost, 0) +
        COALESCE(create_app_bsky_graph_follow, 0) +
        COALESCE(create_app_bsky_graph_block, 0)
    ) AS exposure_created_total,

    -- exposure ricevuta: attenzione/interazione dagli altri
    (
        COALESCE(derived_followed, 0) +
        COALESCE(derived_liked, 0) +
        COALESCE(derived_reposted, 0)
    ) AS exposure_received_total,

    -- blocchi ricevuti per post
    CASE
        WHEN COALESCE(create_app_bsky_feed_post, 0) > 0
        THEN derived_blocked * 1.0 / create_app_bsky_feed_post
        ELSE NULL
    END AS blocked_per_post,

    -- blocchi ricevuti per attività totale
    CASE
        WHEN (
            COALESCE(create_app_bsky_feed_post, 0) +
            COALESCE(create_app_bsky_feed_like, 0) +
            COALESCE(create_app_bsky_feed_repost, 0) +
            COALESCE(create_app_bsky_graph_follow, 0) +
            COALESCE(create_app_bsky_graph_block, 0)
        ) > 0
        THEN derived_blocked * 1.0 / (
            COALESCE(create_app_bsky_feed_post, 0) +
            COALESCE(create_app_bsky_feed_like, 0) +
            COALESCE(create_app_bsky_feed_repost, 0) +
            COALESCE(create_app_bsky_graph_follow, 0) +
            COALESCE(create_app_bsky_graph_block, 0)
        )
        ELSE NULL
    END AS blocked_per_activity

FROM df
""")

In [ ]:
# Analisi delle correlazioni tra exposure e blocchi ricevuti
#Questa query misura la relazione lineare tra diverse proxy di esposizione e il numero di blocchi ricevuti. 
# Nell’output, corr_created_exposure_blocked indica la correlazione tra l’attività totale prodotta dall’account 
# (exposure_created_total) e i blocchi ricevuti (derived__blocked): più è vicina a 1, 
# più i blocchi crescono insieme all’attività. corr_received_exposure_blocked misura invece la correlazione tra 
# l’attenzione ricevuta dall’account (exposure_received_total) e i blocchi ricevuti. corr_posts_blocked mostra la
# correlazione specifica tra numero di post creati e blocchi ricevuti.Questo è solo un primo controllo 
# sintetico: se questi valori sono alti e positivi, significa che l’esposizione conta molto nel determinare quanti
#  blocchi un account riceve
res = con.execute("""
SELECT
    CORR(exposure_created_total, derived_blocked) AS corr_created_exposure_blocked,
    CORR(exposure_received_total, derived_blocked) AS corr_received_exposure_blocked,
    CORR(create_app_bsky_feed_post, derived_blocked) AS corr_posts_blocked
FROM exposure_base
""").df()

res

,corr_created_exposure_blocked,corr_received_exposure_blocked,corr_posts_blocked
0,0.343958,0.268298,0.301548


In [121]:
# Analisi della distribuzione dei blocchi ricevuti in base all'exposure prodotta
#Questa query divide gli account in cinque gruppi ordinati per esposizione e confronta i blocchi ricevuti nei d
# iversi gruppi. Nell’output, exposure_quintile indica il livello di esposizione: 1 corrisponde agli account meno esposti,
#  5 ai più esposti. n_accounts è il numero di account presenti in quel gruppo. avg_exposure è il valore medio di 
# esposizione nel quintile, quindi serve a verificare che i gruppi siano effettivamente ordinati per attività. 
# avg_blocked_received è il numero medio di blocchi ricevuti dagli account di quel gruppo, mentre median_blocked_received
#  è il valore mediano, utile perché meno sensibile a pochi casi estremi. Questa query si usa per vedere se i blocchi 
# ricevuti aumentano in modo sistematico passando da account poco esposti ad account molto esposti
res = con.execute("""
WITH ranked AS (
    SELECT
        *,
        NTILE(5) OVER (ORDER BY exposure_created_total) AS exposure_quintile
    FROM exposure_base
)
SELECT
    exposure_quintile,
    COUNT(*) AS n_accounts,
    AVG(exposure_created_total) AS avg_exposure,
    AVG(derived_blocked) AS avg_blocked_received,
    MEDIAN(derived_blocked) AS median_blocked_received
FROM ranked
GROUP BY 1
ORDER BY 1
""").df()

res

,exposure_quintile,n_accounts,avg_exposure,avg_blocked_received,median_blocked_received
0,1,38712,34.571270,1.487368,0.0
1,2,38711,94.587792,2.314123,1.0
2,3,38711,222.065072,3.654646,1.0
3,4,38711,599.623492,6.298933,2.0
4,5,38711,4888.034047,22.686110,7.0


In [122]:
#Questa query calcola i blocchi ricevuti in forma normalizzata, cioè rapportati all’attività dell’account. 
# Nell’output, avg_blocked_per_post è la media dei blocchi ricevuti per post creato, mentre median_blocked_per_post 
# è la mediana dello stesso rapporto. avg_blocked_per_activity è la media dei blocchi ricevuti rispetto all’attività
#  totale dell’account, e median_blocked_per_activity è la mediana corrispondente. Le medie servono a descrivere il 
# livello medio del fenomeno, mentre le mediane aiutano a capire il comportamento “tipico” evitando che pochi account
#  estremi distorcano il risultato. Questa query si usa per verificare se il segnale dei blocchi resta alto anche dopo 
# aver corretto per esposizione.
res = con.execute("""
SELECT
    AVG(blocked_per_post) AS avg_blocked_per_post,
    MEDIAN(blocked_per_post) AS median_blocked_per_post,
    AVG(blocked_per_activity) AS avg_blocked_per_activity,
    MEDIAN(blocked_per_activity) AS median_blocked_per_activity
FROM exposure_base
WHERE exposure_created_total > 0
""").df()

res

,avg_blocked_per_post,median_blocked_per_post,avg_blocked_per_activity,median_blocked_per_activity
0,0.097241,0.019231,0.021572,0.002924


In [123]:
#Questa query osserva come i tassi normalizzati di blocchi ricevuti si distribuiscono nei diversi livelli di esposizione. 
# Nell’output, exposure_quintile indica ancora il gruppo di esposizione, da 1 a 5. n_accounts è il numero di account 
# nel gruppo. avg_blocked_per_activity mostra il numero medio di blocchi ricevuti per unità di attività totale, 
# mentre median_blocked_per_activity ne mostra il valore mediano. Se questi valori restano simili tra quintili, 
# significa che una volta normalizzato il fenomeno l’effetto dell’esposizione si riduce molto; se invece continuano a 
# crescere nei quintili alti, allora l’esposizione non basta da sola a spiegare il numero di blocchi ricevuti.

res = con.execute("""
WITH ranked AS (
    SELECT
        *,
        NTILE(5) OVER (ORDER BY exposure_created_total) AS exposure_quintile
    FROM exposure_base
    WHERE exposure_created_total > 0
)
SELECT
    exposure_quintile,
    COUNT(*) AS n_accounts,
    AVG(blocked_per_activity) AS avg_blocked_per_activity,
    MEDIAN(blocked_per_activity) AS median_blocked_per_activity
FROM ranked
GROUP BY 1
ORDER BY 1
""").df()

res

,exposure_quintile,n_accounts,avg_blocked_per_activity,median_blocked_per_activity
0,1,38712,0.048788,0.000000
1,2,38711,0.025258,0.007194
2,3,38711,0.016759,0.005348
3,4,38711,0.010926,0.003906
4,5,38711,0.006125,0.002362


In [124]:
#Questa query confronta due tipi diversi di esposizione per capire quale pesa di più rispetto ai blocchi ricevuti. 
# Nell’output, corr_created indica la correlazione tra esposizione prodotta dall’account (exposure_created_total) e
#  blocchi ricevuti. corr_received indica la correlazione tra esposizione ricevuta (exposure_received_total) e 
# blocchi ricevuti. corr_posts misura in modo specifico la relazione tra numero di post e blocchi ricevuti. 
# corr_followed, corr_liked e corr_reposted mostrano invece quanto i blocchi ricevuti sono associati, rispettivamente, 
# all’essere seguiti, likati o repostati da altri. Questa query si usa per capire se i blocchi dipendono soprattutto 
# da quanto l’account agisce direttamente oppure da quanta attenzione riceve nel network.
res = con.execute("""
SELECT
    CORR(exposure_created_total, derived_blocked) AS corr_created,
    CORR(exposure_received_total, derived_blocked) AS corr_received,
    CORR(create_app_bsky_feed_post, derived_blocked) AS corr_posts,
    CORR(derived_followed, derived_blocked) AS corr_followed,
    CORR(derived_liked, derived_blocked) AS corr_liked,
    CORR(derived_reposted, derived_blocked) AS corr_reposted
FROM exposure_base
""").df()

res

,corr_created,corr_received,corr_posts,corr_followed,corr_liked,corr_reposted
0,0.343958,0.268298,0.301548,0.270563,0.263493,0.251623
